# Project 1 Phase A — Python Reference for Fixed 3×3 Real-Space Median Filtering

This notebook prepares a replaceable 4D-STEM source, saves the canonical pre-filter benchmark input, and defines the exact fixed `3 × 3` real-space median contract. It contains no C++, CUDA, bindings, build files, or optimization work.


## Scope boundary: scientific preparation versus the performance target

```text
SCIENTIFIC PREPARATION (Python / 4Denoise; outside median-kernel timing)

ripple_data_reduced.npy
    → center-disk alignment fitted from the current array
    → ellipse fit/correction from the current aligned mean pattern
    → median_filter_input

PHASE A PERFORMANCE TARGET

median_filter_input
    → 3×3 median over scan_y and scan_x only
    → filtered output
```

`ripple_data_reduced.npy` is replaceable. The notebook discovers numerical dimensions at runtime; only the symbolic axis contract is stable.


## Paths, imports, versions, and provenance

Run this notebook from its Project 1 directory. The default project layout places `ripple_data_reduced.npy` one directory above it, `reference_data` beside it, and the generated canonical input under `benchmark_data`. Any Python environment is acceptable if it provides the dependencies and makes the corrected `fourdenoise` module importable.


In [ ]:
from pathlib import Path
import gc
import hashlib
import importlib.util
import io
import inspect
import platform
import time

from IPython.display import Image as DisplayImage, display
import numpy as np
from PIL import Image as PILImage, ImageDraw
import scipy
import skimage


NOTEBOOK_NAME = "python_reference_2d_median_filter.ipynb"
AXIS_ORDER = ("scan_y", "scan_x", "detector_y", "detector_x")
PROJECT_DIR = Path.cwd().resolve()
FOURDENOISE_REPO_CANDIDATE = (PROJECT_DIR.parents[1] / "4denoise_git").resolve()
FOURDENOISE_MODULE_CANDIDATE = FOURDENOISE_REPO_CANDIDATE / "fourdenoise.py"

if FOURDENOISE_MODULE_CANDIDATE.exists():
    module_spec = importlib.util.spec_from_file_location(
        "fourdenoise_project1",
        FOURDENOISE_MODULE_CANDIDATE,
    )
    if module_spec is None or module_spec.loader is None:
        raise ImportError(f"Cannot load {FOURDENOISE_MODULE_CANDIDATE}")
    fd = importlib.util.module_from_spec(module_spec)
    module_spec.loader.exec_module(fd)
else:
    import fourdenoise as fd

DATA_PATH = (PROJECT_DIR.parent / "ripple_data_reduced.npy").resolve()
REFERENCE_DIR = (PROJECT_DIR / "reference_data").resolve()
BENCHMARK_DIR = (PROJECT_DIR / "benchmark_data").resolve()
BENCHMARK_INPUT_PATH = BENCHMARK_DIR / "median_filter_input.npy"
DATASET_MANIFEST_PATH = BENCHMARK_DIR / "DATASET.md"
FOURDENOISE_MODULE_PATH = Path(fd.__file__).resolve()
alignment_parameters = set(inspect.signature(fd.HyperData.alignment).parameters)
required_alignment_parameters = {
    "method",
    "fit_radius",
    "radius_range",
    "radius_step",
    "iterations",
    "search_radius",
    "enforce_square",
}
if not required_alignment_parameters.issubset(alignment_parameters):
    missing = sorted(required_alignment_parameters - alignment_parameters)
    raise ImportError(
        f"Imported incompatible fourdenoise module {FOURDENOISE_MODULE_PATH}; "
        f"HyperData.alignment is missing parameters {missing}. "
        "Make the current corrected 4Denoise repository importable."
    )
if not hasattr(fd.HyperData, "denoise"):
    raise ImportError(
        f"Imported incompatible fourdenoise module {FOURDENOISE_MODULE_PATH}; "
        "HyperData.denoise is unavailable."
    )

if not DATA_PATH.exists():
    raise FileNotFoundError(DATA_PATH)
if not REFERENCE_DIR.exists():
    raise FileNotFoundError(REFERENCE_DIR)
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)

def file_sha256(path, chunk_bytes=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_bytes), b""):
            digest.update(chunk)
    return digest.hexdigest()

source_hashes = {
    "reduced_data": file_sha256(DATA_PATH),
    "fourdenoise_module": file_sha256(FOURDENOISE_MODULE_PATH),
}

print(f"Project directory: {PROJECT_DIR}")
print(f"Python {platform.python_version()}")
print(
    f"NumPy {np.__version__}; SciPy {scipy.__version__}; "
    f"scikit-image {skimage.__version__}"
)
print(f"4Denoise module: {FOURDENOISE_MODULE_PATH}")
for name, digest in source_hashes.items():
    print(f"{name} SHA-256: {digest}")


## Pipeline used in this notebook

The explicit upstream axis contract is `(scan_y, scan_x, detector_y, detector_x)`. Shape does not establish axis meaning; that interpretation comes from the 4Denoise workflow that produced the replaceable source.

This notebook fits current alignment and ellipse quantities, saves the prepared input, and then applies the fixed median through the public `HyperData.denoise` API. Physical-space calibration, dose calculations, and virtual-image displays are not required for this array contract.


## Load and inspect the replaceable reduced dataset

The source is loaded into writable RAM because alignment writes through detector views in the installed image-warp stack. Loading does not modify the NPY file. The checks below discover dimensions and metadata dynamically and fail clearly for a nonnumeric, empty, nonfinite, or non-4D input.


In [ ]:
load_started = time.perf_counter()
reduced_source = np.load(DATA_PATH, allow_pickle=False)
load_seconds = time.perf_counter() - load_started

if reduced_source.ndim != 4:
    raise ValueError(
        "Expected a 4D array with axis semantics "
        f"{AXIS_ORDER}; received shape {reduced_source.shape}"
    )
if any(length == 0 for length in reduced_source.shape):
    raise ValueError(f"All four dimensions must be nonempty: {reduced_source.shape}")
if not np.issubdtype(reduced_source.dtype, np.number):
    raise TypeError(f"Expected numeric data, received dtype {reduced_source.dtype}")

source_shape = tuple(int(length) for length in reduced_source.shape)
scan_y, scan_x, detector_y, detector_x = source_shape
source_dtype = reduced_source.dtype
source_strides = reduced_source.strides
source_nbytes = reduced_source.nbytes
source_c_contiguous = reduced_source.flags.c_contiguous
source_f_contiguous = reduced_source.flags.f_contiguous

print(f"current loaded shape: {source_shape}")
print(f"dtype: {source_dtype}")
print(f"axis order (from upstream provenance): {AXIS_ORDER}")
print(f"logical bytes: {source_nbytes:,} ({source_nbytes / 2**20:.3f} MiB)")
print(f"NPY file bytes: {DATA_PATH.stat().st_size:,}")
print(f"strides: {source_strides}")
print(f"C contiguous: {source_c_contiguous}")
print(f"F contiguous: {source_f_contiguous}")
print(f"writable in-memory load: {reduced_source.flags.writeable}")
print(f"load time: {load_seconds:.6f} s")


## Axis semantics

```text
axis 0          axis 1          axis 2              axis 3
scan_y          scan_x          detector_y (k_y)    detector_x (k_x)
real space      real space      reciprocal space    reciprocal space
```

`data[:, :, qy, qx]` is one real-space scan image at a fixed detector coordinate. `data[sy, sx, :, :]` is one diffraction pattern. The NPY format does not store axis labels, so a replacement must retain this upstream convention; the notebook never guesses it from dimension sizes.


In [ ]:
def finite_summary(array):
    nan_count = posinf_count = neginf_count = 0
    finite_min = np.inf
    finite_max = -np.inf
    for scan_index in range(array.shape[0]):
        block = np.asarray(array[scan_index])
        nan_count += int(np.isnan(block).sum())
        posinf_count += int(np.isposinf(block).sum())
        neginf_count += int(np.isneginf(block).sum())
        finite = np.isfinite(block)
        if finite.any():
            values = block[finite]
            finite_min = min(finite_min, float(values.min()))
            finite_max = max(finite_max, float(values.max()))
    return {
        "nan_count": nan_count,
        "posinf_count": posinf_count,
        "neginf_count": neginf_count,
        "finite_min": finite_min,
        "finite_max": finite_max,
    }

input_inspection_started = time.perf_counter()
input_finite_summary = finite_summary(reduced_source)
input_inspection_seconds = time.perf_counter() - input_inspection_started
if any(
    input_finite_summary[name] != 0
    for name in ("nan_count", "posinf_count", "neginf_count")
):
    raise ValueError(f"Median reference requires finite data: {input_finite_summary}")
print(input_finite_summary)
print(f"complete finite-value scan: {input_inspection_seconds:.3f} s")


## Scientific preparation stage 1 — center-disk alignment

`alignment(method='disk', ...)` fits a filled-disk radius and reference center from the current mean diffraction pattern, template-matches a center at every scan position, determines a common centered detector crop, and applies subpixel shifts. Two iterations refit residual offsets.

Scan dimensions are checked to remain unchanged. Detector dimensions are allowed to change because the largest common valid crop is data-dependent; no numerical output shape is assumed.


In [ ]:
raw_hyperdata = fd.HyperData(reduced_source)

alignment_started = time.perf_counter()
aligned_data = raw_hyperdata.alignment(
    method="disk",
    fit_radius=True,
    radius_range=[0.1, 3],
    radius_step=0.01,
    iterations=2,
    search_radius=2.5,
    enforce_square=False,
)
alignment_seconds = time.perf_counter() - alignment_started
alignment_metadata = dict(aligned_data.center_beam_metadata)
aligned_shape = tuple(int(length) for length in aligned_data.shape)

if aligned_shape[:2] != source_shape[:2]:
    raise AssertionError(
        f"Alignment changed scan axes: source {source_shape[:2]}, aligned {aligned_shape[:2]}"
    )

print(
    "alignment parameters: method='disk', fit_radius=True, "
    "radius_range=[0.1, 3], radius_step=0.01, iterations=2, "
    "search_radius=2.5, enforce_square=False"
)
print(f"aligned shape: {aligned_shape}")
print(f"aligned dtype: {aligned_data.dtype}")
print(f"aligned strides: {aligned_data.array.strides}")
print(f"aligned C contiguous: {aligned_data.array.flags.c_contiguous}")
print(f"alignment time: {alignment_seconds:.6f} s")
print(f"selected radius: {alignment_metadata['radius_px']:.12g} px")
print(f"reference center: {alignment_metadata['reference_center_px']}")
print(f"target center: {alignment_metadata['target_center_px']}")
print(f"final mean center: {alignment_metadata['mean_fit_center_px']}")
print(f"final center std: {alignment_metadata['std_fit_center_px']}")

del raw_hyperdata, reduced_source
gc.collect()


## Scientific preparation stage 2 — current ellipse fit and correction

The established annulus is `43.5 ≤ radius ≤ 51.5` detector pixels. Before it is reused, the notebook verifies that the complete annulus fits inside the **current aligned detector grid** and that the current mean pattern yields a finite ellipse with fitted semiaxes inside that annulus. The correction is then applied through the public API.

These checks can validate geometry in pixel coordinates, but an unlabeled NPY cannot reveal upstream detector resampling. If a replacement changes detector pixel size, shifts the physical origin, or removes the calibrated ring, `r` and `R` must be re-established scientifically even if the numerical fit happens to succeed.


In [ ]:
aligned_detector_shape = aligned_shape[-2:]
aligned_center_y = aligned_detector_shape[0] // 2
aligned_center_x = aligned_detector_shape[1] // 2
complete_annulus_radius = min(
    aligned_center_y,
    aligned_detector_shape[0] - 1 - aligned_center_y,
    aligned_center_x,
    aligned_detector_shape[1] - 1 - aligned_center_x,
)
if complete_annulus_radius < 51.5:
    raise ValueError(
        "The established outer ellipse-fit radius (51.5 detector pixels) "
        f"does not fit in the current aligned detector shape {aligned_detector_shape}. "
        "Re-establish detector calibration and r/R for this replacement source."
    )

mean_dp_before_ellipse = aligned_data.get_dp("mean").array
ellipse_before = aligned_data._extract_ellipse(
    mean_dp_before_ellipse,
    aligned_center_y,
    aligned_center_x,
    43.5,
    51.5,
)
ellipse_before_values = tuple(float(value) for value in ellipse_before)
if not np.isfinite(ellipse_before_values).all():
    raise ValueError(f"Current ellipse fit is nonfinite: {ellipse_before_values}")
if not all(43.5 <= axis <= 51.5 for axis in ellipse_before_values[1:]):
    raise ValueError(
        "Current fitted ellipse axes fall outside the established annulus: "
        f"{ellipse_before_values}. Reassess detector calibration and r/R."
    )

ellipse_started = time.perf_counter()
elliptically_corrected = aligned_data.fix_elliptical_distortions(
    r=43.5,
    R=51.5,
    interp_method="linear",
    return_fix=True,
)
ellipse_seconds = time.perf_counter() - ellipse_started

corrected_shape = tuple(int(length) for length in elliptically_corrected.shape)
mean_dp_after_ellipse = elliptically_corrected.get_dp("mean").array
ellipse_after = elliptically_corrected._extract_ellipse(
    mean_dp_after_ellipse,
    corrected_shape[-2] // 2,
    corrected_shape[-1] // 2,
    43.5,
    51.5,
)
ellipse_after_values = tuple(float(value) for value in ellipse_after)

if corrected_shape != aligned_shape:
    raise AssertionError(
        f"Ellipse correction unexpectedly changed shape: {aligned_shape} -> {corrected_shape}"
    )
if not np.isfinite(ellipse_after_values).all():
    raise ValueError(f"Post-correction ellipse fit is nonfinite: {ellipse_after_values}")

print(
    "elliptical-correction parameters: r=43.5, R=51.5, "
    "interp_method='linear', return_fix=True"
)
print("ellipse correction path: public 4Denoise API")
print(f"ellipse before (angle rad, a px, b px): {ellipse_before_values}")
print(f"ellipse after  (angle rad, a px, b px): {ellipse_after_values}")
print(f"corrected shape: {corrected_shape}")
print(f"corrected dtype: {elliptically_corrected.dtype}")
print(f"corrected strides: {elliptically_corrected.array.strides}")
print(f"corrected C contiguous: {elliptically_corrected.array.flags.c_contiguous}")
print(f"elliptical-correction time: {ellipse_seconds:.6f} s")


## Canonical pre-filter array

`median_filter_input` is the scientifically prepared 4D array immediately before either median operation. Its general shape is `(scan_y_processed, scan_x_processed, detector_y_processed, detector_x_processed)`; the current numerical shape and dtype are runtime results, not project invariants.


In [ ]:
median_filter_input = np.ascontiguousarray(elliptically_corrected.array)
preprocessing_seconds = alignment_seconds + ellipse_seconds

if median_filter_input.ndim != 4:
    raise AssertionError(f"Prepared array is not 4D: {median_filter_input.shape}")
if median_filter_input.shape[:2] != source_shape[:2]:
    raise AssertionError(
        "Scientific preparation changed scan-grid dimensions: "
        f"{source_shape[:2]} -> {median_filter_input.shape[:2]}"
    )
if not np.issubdtype(median_filter_input.dtype, np.number):
    raise TypeError(f"Prepared dtype is not numeric: {median_filter_input.dtype}")
if not median_filter_input.flags.c_contiguous:
    raise AssertionError("Canonical prepared input must be C-contiguous")

prepared_finite_summary = finite_summary(median_filter_input)
if any(
    prepared_finite_summary[name] != 0
    for name in ("nan_count", "posinf_count", "neginf_count")
):
    raise ValueError(f"Prepared input must be finite: {prepared_finite_summary}")

scan_y_processed, scan_x_processed, detector_y_processed, detector_x_processed = (
    median_filter_input.shape
)
print("variable: median_filter_input")
print(f"current prepared shape: {median_filter_input.shape}")
print(f"dtype: {median_filter_input.dtype}")
print(f"axis order: {AXIS_ORDER}")
print(f"strides: {median_filter_input.strides}")
print(f"C contiguous: {median_filter_input.flags.c_contiguous}")
print(f"finite-value summary: {prepared_finite_summary}")
print(f"numeric preprocessing time: {preprocessing_seconds:.6f} s")

del mean_dp_before_ellipse, mean_dp_after_ellipse, aligned_data, elliptically_corrected
gc.collect()


## Save and verify the canonical benchmark input

Both Project 1 notebooks produce the same `median_filter_input` from the current source. The save is replaced only when its bytes differ. The file is then reloaded, checked slab by slab for bitwise identity, hashed, and documented in `benchmark_data/DATASET.md`. No filtered full-array output is saved.


In [ ]:
def arrays_are_bitwise_equal_by_scan(left, right):
    if left.shape != right.shape or left.dtype != right.dtype:
        return False
    for scan_index in range(left.shape[0]):
        left_bytes = np.asarray(left[scan_index]).view(np.uint8)
        right_bytes = np.asarray(right[scan_index]).view(np.uint8)
        if not np.array_equal(left_bytes, right_bytes):
            return False
    return True

def save_npy_if_changed(path, array):
    if path.exists():
        existing = np.load(path, allow_pickle=False, mmap_mode="r")
        unchanged = arrays_are_bitwise_equal_by_scan(existing, array)
        del existing
        if unchanged:
            return False
    temporary_path = path.with_name(path.name + ".tmp")
    with temporary_path.open("wb") as stream:
        np.save(stream, array, allow_pickle=False)
    temporary_path.replace(path)
    return True

benchmark_written = save_npy_if_changed(BENCHMARK_INPUT_PATH, median_filter_input)
benchmark_reloaded = np.load(
    BENCHMARK_INPUT_PATH,
    allow_pickle=False,
    mmap_mode="r",
)

if benchmark_reloaded.shape != median_filter_input.shape:
    raise AssertionError("Saved benchmark shape differs from the in-memory prepared array")
if benchmark_reloaded.dtype != median_filter_input.dtype:
    raise AssertionError("Saved benchmark dtype differs from the in-memory prepared array")
if not benchmark_reloaded.flags.c_contiguous:
    raise AssertionError("Saved benchmark input must be C-contiguous")
if not all(
    np.isfinite(np.asarray(benchmark_reloaded[scan_index])).all()
    for scan_index in range(benchmark_reloaded.shape[0])
):
    raise ValueError("Saved benchmark input contains NaN or infinity")
if not arrays_are_bitwise_equal_by_scan(benchmark_reloaded, median_filter_input):
    raise AssertionError("Saved benchmark input is not bit-for-bit identical to memory")

benchmark_hash = file_sha256(BENCHMARK_INPUT_PATH)
benchmark_file_bytes = BENCHMARK_INPUT_PATH.stat().st_size

known_scan_only_source_hash = (
    "3fa102dbc58d1dd4e8cf2caf22cda07b4897edb2dbfec5d921a1c11c42fd30fd"
)
if source_hashes["reduced_data"] == known_scan_only_source_hash:
    detector_provenance = (
        "For this source hash, inspection of the upstream 4Denoise workflow shows "
        "a scan-space crop (`ylim=(0, 85)`, `xlim=(2, 37)`) before saving. "
        "The detector axes were not cropped or resampled, so the established "
        "43.5–51.5 pixel annulus retains its original detector-pixel meaning."
    )
else:
    detector_provenance = (
        "The NPY file alone cannot establish whether detector pixels were cropped, "
        "shifted, or resampled upstream. This run confirmed that the complete "
        "43.5–51.5 pixel annulus fits in the aligned detector grid and that a finite "
        "ellipse was fitted from the current mean diffraction pattern. If the "
        "replacement changed detector sampling or removed the calibrated ring, "
        "re-establish `r` and `R` scientifically before using this artifact."
    )

dataset_manifest = f"""# Project 1 Canonical Benchmark Input

## Purpose

`median_filter_input.npy` is the common scientifically prepared real-data input for future fixed C++, fixed CUDA, adaptive C++, and adaptive CUDA median filtering. It is center-aligned and elliptically corrected, but it is not itself median-filtered.

Small deterministic correctness cases remain under `reference_data/`; this file is the realistic development and performance input.

## Source

- Path: `{DATA_PATH}`
- Current SHA-256: `{source_hashes['reduced_data']}`
- Current runtime shape: `{source_shape}`
- Current dtype: `{source_dtype}`
- Axis interpretation: `{AXIS_ORDER}` (the established upstream 4Denoise convention, not inferred from dimensions)
- C contiguous: `{source_c_contiguous}`
- F contiguous: `{source_f_contiguous}`
- Strides: `{source_strides}` bytes
- Finite: `True`
- Current value range: `[{input_finite_summary['finite_min']}, {input_finite_summary['finite_max']}]`
- Logical bytes: `{source_nbytes}`
- NPY file bytes: `{DATA_PATH.stat().st_size}`

The source file is replaceable and may be overwritten by another reduced version. Every numerical dimension and calibration statement below describes only the artifact generated by this run; it is not a permanent project shape.

## Scientific preprocessing

1. `HyperData.alignment(method='disk', fit_radius=True, radius_range=[0.1, 3], radius_step=0.01, iterations=2, search_radius=2.5, enforce_square=False)`
   - Current fitted disk radius: `{float(alignment_metadata['radius_px'])}` pixels
   - Current reference center: `{tuple(alignment_metadata['reference_center_px'])}`
   - Current target center: `{tuple(alignment_metadata['target_center_px'])}`
   - Current final mean fitted center: `{tuple(alignment_metadata['mean_fit_center_px'])}`
   - Current final fitted-center standard deviation: `{tuple(alignment_metadata['std_fit_center_px'])}`
   - Current aligned shape: `{aligned_shape}`
2. `HyperData.fix_elliptical_distortions(r=43.5, R=51.5, interp_method='linear', return_fix=True)`
   - Current ellipse before correction `(angle_rad, a_px, b_px)`: `{ellipse_before_values}`
   - Current ellipse after correction `(angle_rad, a_px, b_px)`: `{ellipse_after_values}`
   - Current correction output shape: `{median_filter_input.shape}`

{detector_provenance}

Alignment derives disk centers from the current input. Ellipse parameters are fitted from the current aligned mean diffraction pattern. The notebook uses the normal package API; it contains no affine-transform workaround.

## Array contract

The general axis contract is `(scan_y, scan_x, detector_y, detector_x)`. Numerical dimensions are discovered at runtime. Scan axes retain their upstream grid semantics. Alignment can reduce detector extents to a common valid crop; ellipse correction preserves the aligned shape.

Current generated artifact:

- Shape: `{median_filter_input.shape}`
- Dtype: `{median_filter_input.dtype}`
- Axis order: `{AXIS_ORDER}`
- C contiguous: `{median_filter_input.flags.c_contiguous}`
- F contiguous: `{median_filter_input.flags.f_contiguous}`
- Strides: `{median_filter_input.strides}` bytes
- Finite: `True`
- Value range: `[{prepared_finite_summary['finite_min']}, {prepared_finite_summary['finite_max']}]`
- Logical bytes: `{median_filter_input.nbytes}`
- NPY file bytes: `{benchmark_file_bytes}`

## Provenance

- Source SHA-256: `{source_hashes['reduced_data']}`
- Benchmark SHA-256: `{benchmark_hash}`
- Imported 4Denoise module: `{FOURDENOISE_MODULE_PATH}`
- Imported module SHA-256: `{source_hashes['fourdenoise_module']}`
- Scientific procedure: the explicit public 4Denoise calls and runtime-derived values documented above
- Regeneration notebooks: `python_reference_2d_median_filter.ipynb` and `python_reference_adaptive_median_filter.ipynb`
- Notebook that last generated/verified this manifest: `{NOTEBOOK_NAME}`

## Verification

The saved NPY was reloaded as a C-contiguous array and checked for exact shape and dtype, finite values, and bit-for-bit equality with the in-memory `median_filter_input`, one scan slab at a time.

## Regeneration

If `ripple_data_reduced.npy` is overwritten, rerun either Project 1 reference notebook from top to bottom. The notebook refits current alignment and ellipse quantities, regenerates `median_filter_input.npy`, reloads and verifies it, recomputes both SHA-256 values, and updates this file. A replacement that changes detector sampling requires scientific review of the 43.5–51.5 pixel annulus; geometric fit checks alone cannot recover missing physical provenance.

## Performance boundary

Project 1 median benchmarks begin from `median_filter_input.npy`. Source loading, center alignment, ellipse fitting, and elliptical correction are excluded from median-kernel timing. Full filtered reference outputs are intentionally not stored here.
"""
DATASET_MANIFEST_PATH.write_text(dataset_manifest, encoding="utf-8")

print(f"canonical benchmark rewritten: {benchmark_written}")
print(f"canonical path: {BENCHMARK_INPUT_PATH}")
print(f"canonical shape/dtype: {benchmark_reloaded.shape}, {benchmark_reloaded.dtype}")
print(f"canonical strides: {benchmark_reloaded.strides}")
print(f"canonical file bytes: {benchmark_file_bytes:,}")
print(f"canonical SHA-256: {benchmark_hash}")
print("Reloaded canonical array is finite and bit-for-bit identical to memory.")


## Phase A performance target — exact `3 × 3` median

The result is produced by the public `HyperData.denoise` API. With `domain='real'`, the median is taken over axes 0 and 1 while every detector coordinate remains independent. The call explicitly specifies a centered `3 × 3` window, SciPy `reflect` boundaries, `origin=0`, and `cval=0.0` (ignored in reflect mode). The operation returns a new same-shape, same-dtype array and does not mutate its input.


## Frozen fixed-median correctness fixture

The small fixture under `reference_data/` is a frozen, already-preprocessed correctness contract. Its numerical shape and historical source slices are fixture-specific metadata, not assumptions about the replaceable current source. This notebook does not regenerate it; it reloads it and recomputes the expected output through the public API.


In [ ]:
reference_input_path = REFERENCE_DIR / "reference_input.npy"
reference_output_path = REFERENCE_DIR / "reference_output_python.npy"
reference_spec_path = REFERENCE_DIR / "REFERENCE_DATA.md"

reference_input = np.load(reference_input_path, allow_pickle=False)
saved_reference_output = np.load(reference_output_path, allow_pickle=False)
reference_input_before = reference_input.copy()

fixture_started = time.perf_counter()
reference_output = fd.HyperData(reference_input).denoise(
    method="median",
    domain="real",
    window_size=3,
    mode="reflect",
    cval=0.0,
    origin=0,
    return_array=True,
)
fixture_filter_seconds = time.perf_counter() - fixture_started

if reference_input.shape != saved_reference_output.shape:
    raise AssertionError("Frozen fixed fixture input/output shapes differ")
if reference_input.dtype != saved_reference_output.dtype:
    raise AssertionError("Frozen fixed fixture input/output dtypes differ")
if not (reference_input.flags.c_contiguous and saved_reference_output.flags.c_contiguous):
    raise AssertionError("Frozen fixed fixtures must be C-contiguous")
if not np.array_equal(reference_input.view(np.uint64), reference_input_before.view(np.uint64)):
    raise AssertionError("Public median API mutated the frozen fixture input")
if not np.array_equal(reference_output.view(np.uint64), saved_reference_output.view(np.uint64)):
    raise AssertionError("Frozen fixed fixture no longer matches the public API")

reference_input_hash = file_sha256(reference_input_path)
reference_output_hash = file_sha256(reference_output_path)
assert reference_input_hash == "844a4bfa1dbe13b017d2972f8fb05b91ce85447d9f0454af24acf3fee157fd6b"
assert reference_output_hash == "125c1458ff38f2b8fd741ce9217e0e1f974c9d24d4d96a73da558f0810b0ace2"

print(f"frozen fixture shape/dtype: {reference_input.shape}, {reference_input.dtype}")
print(f"fixture filter time: {fixture_filter_seconds:.6f} s")
print(f"input SHA-256:  {reference_input_hash}")
print(f"output SHA-256: {reference_output_hash}")
print("Frozen expected output matches a fresh public 4Denoise.denoise result bit for bit.")


## Full current-data median baseline

This cell times one complete fixed-median pass over the current `median_filter_input`. It excludes source loading, scientific preprocessing, diagnostics, visualization, and file I/O. No full filtered output is saved.


In [ ]:
full_filter_started = time.perf_counter()
full_filtered_output = fd.HyperData(median_filter_input).denoise(
    method="median",
    domain="real",
    window_size=3,
    mode="reflect",
    cval=0.0,
    origin=0,
    return_array=True,
)
full_filter_seconds = time.perf_counter() - full_filter_started

if full_filtered_output.shape != median_filter_input.shape:
    raise AssertionError("Fixed median changed the array shape")
if full_filtered_output.dtype != median_filter_input.dtype:
    raise AssertionError("Fixed median changed the array dtype")
if not full_filtered_output.flags.c_contiguous:
    raise AssertionError("Fixed median output is not C-contiguous")
if not arrays_are_bitwise_equal_by_scan(benchmark_reloaded, median_filter_input):
    raise AssertionError("Fixed median mutated median_filter_input")

relevant_workflow_seconds = load_seconds + preprocessing_seconds + full_filter_seconds
print(f"current source shape: {source_shape}")
print(f"current prepared shape: {median_filter_input.shape}")
print(f"writable load time: {load_seconds:.6f} s")
print(f"alignment time: {alignment_seconds:.6f} s")
print(f"elliptical-correction time: {ellipse_seconds:.6f} s")
print(f"total numeric preprocessing time: {preprocessing_seconds:.6f} s")
print(f"full 3×3 median-filter computation: {full_filter_seconds:.6f} s")
print(f"load + preprocessing + median: {relevant_workflow_seconds:.6f} s")
print(f"frozen correctness fixture median: {fixture_filter_seconds:.6f} s")


## Compact visual validation

The display coordinate is derived from the current alignment target and clipped to the current processed detector bounds. The panels show one full current scan image before filtering, after filtering, and their signed difference. Display scaling never changes reference arrays.


In [ ]:
fitted_target = alignment_metadata["target_center_px"]
detector_coordinate = (
    int(np.clip(round(float(fitted_target[0])), 0, detector_y_processed - 1)),
    int(np.clip(round(float(fitted_target[1])), 0, detector_x_processed - 1)),
)
before_image = median_filter_input[:, :, detector_coordinate[0], detector_coordinate[1]]
after_image = full_filtered_output[:, :, detector_coordinate[0], detector_coordinate[1]]
difference_image = after_image - before_image

display_low, display_high = np.percentile(before_image, (1, 99))
difference_limit = float(np.percentile(np.abs(difference_image), 99))

def grayscale_uint8(image, low, high):
    if not high > low:
        return np.zeros(image.shape, dtype=np.uint8)
    scaled = np.clip((image - low) / (high - low), 0.0, 1.0)
    return np.rint(255.0 * scaled).astype(np.uint8)

panels = (
    grayscale_uint8(before_image, display_low, display_high),
    grayscale_uint8(after_image, display_low, display_high),
    grayscale_uint8(difference_image, -difference_limit, difference_limit),
)
labels = ("Prepared input", "After 3x3 median", "After - before")
scale = max(1, min(4, 640 // max(scan_x_processed, 1)))
separator = 8
header = 28
panel_width = panels[0].shape[1] * scale
panel_height = panels[0].shape[0] * scale
canvas = PILImage.new(
    "L",
    (3 * panel_width + 2 * separator, header + panel_height),
    color=255,
)
drawing = ImageDraw.Draw(canvas)
for index, (panel, label) in enumerate(zip(panels, labels)):
    tile = PILImage.fromarray(panel, mode="L").resize(
        (panel_width, panel_height),
        resample=PILImage.Resampling.NEAREST,
    )
    left = index * (panel_width + separator)
    canvas.paste(tile, (left, header))
    drawing.text((left + 4, 7), label, fill=0)

buffer = io.BytesIO()
canvas.save(buffer, format="PNG")
display(DisplayImage(data=buffer.getvalue()))
print(
    f"Detector coordinate {detector_coordinate}; shared display limits "
    f"[{display_low:.4g}, {display_high:.4g}]; difference limit ±{difference_limit:.4g}."
)


# C++ / CUDA Behavioral Contract

## Input

- The median-filter input is the already center-aligned and elliptically corrected `median_filter_input`.
- General shape convention: `(scan_y, scan_x, detector_y, detector_x)`; numerical dimensions are runtime values.
- Axes 0–1 are real-space scan coordinates; axes 2–3 are detector coordinates.
- The reference dtype is the dtype stored in the current canonical benchmark artifact.
- Inputs must be finite and must not be mutated.

Center alignment and ellipse correction are outside initial median-kernel timing.

## Operation

- Apply a centered rectangular `3 × 3` median (9 samples; select the fifth ordered value) over axes 0 and 1 only.
- Treat every `(detector_y, detector_x)` coordinate independently.
- Boundary mode: SciPy `reflect` (half-sample symmetric; edge samples repeat).
- Origin: `0`; `cval=0.0` is ignored under reflect mode.
- Return an independent array with the same shape and dtype.

## Validation

The frozen finite-`float64` fixture requires bit-for-bit equality. Future benchmark comparisons must separately document any dtype-specific criterion.


## Remaining scope

Scientific preparation is parameterized and fitted from the loaded data. The package-level NumPy/OpenCV dimension-order fix handles non-square ellipse correction, and this notebook uses the normal public API without a local workaround.

Formal benchmark methodology, hardware reporting, C++/CUDA implementations, profiling, optimization, bindings, and broader dtype support remain future work.


## Final verification

The final cell rechecks source integrity, the canonical saved input, the frozen fixture, and the public fixed-median operation. It does not save the full filtered output.


In [ ]:
final_source_hash = file_sha256(DATA_PATH)
assert final_source_hash == source_hashes["reduced_data"]

final_benchmark = np.load(BENCHMARK_INPUT_PATH, allow_pickle=False, mmap_mode="r")
assert arrays_are_bitwise_equal_by_scan(final_benchmark, median_filter_input)
assert file_sha256(BENCHMARK_INPUT_PATH) == benchmark_hash

final_recomputed_output = fd.HyperData(reference_input).denoise(
    method="median",
    domain="real",
    window_size=3,
    mode="reflect",
    cval=0.0,
    origin=0,
    return_array=True,
)
assert np.array_equal(
    final_recomputed_output.view(np.uint64),
    saved_reference_output.view(np.uint64),
)
assert reference_spec_path.exists()
assert DATASET_MANIFEST_PATH.exists()

print("Final fixed-reference verification passed.")
print(f"Source unchanged: {final_source_hash}")
print(f"Prepared input: {median_filter_input.shape}, {median_filter_input.dtype}")
print(f"Canonical SHA-256: {benchmark_hash}")
print("Median: window_size=3 over scan axes, reflect, cval=0.0, origin=0")
print(f"Frozen fixture input:  {reference_input_path}")
print(f"Frozen fixture output: {reference_output_path}")
print("Canonical and frozen-fixture checks passed bit for bit.")

del full_filtered_output, final_recomputed_output, final_benchmark, benchmark_reloaded
gc.collect()
